# 🌏 NZ Earthquake Declustering — Method 3: Gaussian Mixture Model (GMM)
### Probabilistic soft-assignment declustering

**Core idea:** Model the feature space as a mixture of Gaussian distributions.
Each Gaussian captures a 'mode' of seismicity. Events get a **probability of belonging to each component**.

**Why GMM for seismicity?**
- Produces **true posterior probabilities** per event (not just hard labels)
- Captures the two-lobe structure naturally (like Zaliapin's T-R plot)
- Interpretable: each Gaussian = a distinct seismic behaviour regime
- Works directly on your pre-computed features
- Fast: fits in minutes even on 380k events

---
### Pipeline
```
Load → Clean → 22 Features → PCA (reduce to 10D) → GMM (n_components=2..20)
     → BIC/AIC selection → Classify components → Event probabilities → Save
```
---
> **Key advantage over KDM:** GMM is fully parametric — each component has a
> mean vector and covariance matrix that you can interpret physically.

## 📦 Step 0 — Install & Import

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
def out(f): return str(OUTPUT_DIR / f)

print('✅ All imports done!')

---
## 📂 Step 1 — Load & Clean Data

In [ ]:
YOUR_CSV = 'som_feature_nz_real_catalog.csv'

df = pd.read_csv(YOUR_CSV, low_memory=False)
t_cols = [f'T{i}' for i in range(1,11)]
r_cols = [f'R{i}' for i in range(1,11)]
required = t_cols + r_cols + ['Mn','bval']

df_clean = df.dropna(subset=required).copy()
if 'i+' in df_clean.columns:
    df_clean = df_clean[df_clean['i+'] > 0].copy()
df_clean = df_clean.reset_index(drop=True)
print(f'Clean shape: {df_clean.shape}')

---
## 🧮 Step 2 — Feature Matrix + PCA

GMM suffers from the **curse of dimensionality** in 22D — covariance matrices become
ill-conditioned. We use PCA to reduce to 10 components while retaining ~95% variance.

This also speeds up fitting dramatically on 380k events.

In [ ]:
feature_cols = t_cols + r_cols + ['Mn','bval']
X = df_clean[feature_cols].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA to 10 components
N_PCA = 10
pca   = PCA(n_components=N_PCA, random_state=42)
X_pca = pca.fit_transform(X_scaled)

var_explained = pca.explained_variance_ratio_.cumsum()
print(f'Feature matrix  : {X_scaled.shape}')
print(f'PCA output      : {X_pca.shape}')
print(f'Variance retained: {var_explained[-1]:.2%}')

# Scree plot
fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].bar(range(1,N_PCA+1), pca.explained_variance_ratio_*100, color='steelblue')
axes[0].set_title('Explained Variance per Component')
axes[0].set_xlabel('PCA Component'); axes[0].set_ylabel('Variance (%)')
axes[1].plot(range(1,N_PCA+1), var_explained*100, 'bo-')
axes[1].axhline(95, color='red', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Variance Explained')
axes[1].set_xlabel('PCA Components'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig(out('GMM_pca_variance.png'), dpi=150)
plt.show()

---
## 🔍 Step 3 — Select Optimal Number of GMM Components

We use **BIC (Bayesian Information Criterion)** and **AIC (Akaike IC)** to find the
optimal number of Gaussian components. Lower = better fit with less overfitting.

| Criterion | What it penalises | Use when |
|---|---|---|
| **BIC** | Model complexity (harder penalty) | Prefer fewer components |
| **AIC** | Model fit (softer penalty) | Prefer more components |

For declustering we want exactly **2 dominant components** (crisis + non-crisis),
but more components can model sub-types (aftershocks vs swarms).

In [ ]:
N_COMPONENTS_RANGE = range(2, 15)   # test 2 to 14 components
COVARIANCE_TYPE    = 'full'         # 'full', 'tied', 'diag', 'spherical'
SUBSAMPLE_BIC      = 50_000         # use subsample for BIC speed

idx_bic  = np.random.choice(len(X_pca), min(SUBSAMPLE_BIC, len(X_pca)), replace=False)
X_bic    = X_pca[idx_bic]

bic_scores, aic_scores = [], []
print('Computing BIC/AIC...')
for n in N_COMPONENTS_RANGE:
    gmm = GaussianMixture(n_components=n, covariance_type=COVARIANCE_TYPE,
                          random_state=42, max_iter=200, n_init=3)
    gmm.fit(X_bic)
    bic_scores.append(gmm.bic(X_bic))
    aic_scores.append(gmm.aic(X_bic))
    print(f'  n={n:2d}  BIC={gmm.bic(X_bic):.0f}  AIC={gmm.aic(X_bic):.0f}')

best_bic_n = list(N_COMPONENTS_RANGE)[np.argmin(bic_scores)]
best_aic_n = list(N_COMPONENTS_RANGE)[np.argmin(aic_scores)]
print(f'\n✅ Best n by BIC: {best_bic_n}')
print(f'✅ Best n by AIC: {best_aic_n}')

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(N_COMPONENTS_RANGE, bic_scores, 'bo-', label='BIC')
ax.plot(N_COMPONENTS_RANGE, aic_scores, 'rs--', label='AIC')
ax.axvline(best_bic_n, color='blue',  linestyle=':', alpha=0.7, label=f'Best BIC = {best_bic_n}')
ax.axvline(best_aic_n, color='red',   linestyle=':', alpha=0.7, label=f'Best AIC = {best_aic_n}')
ax.set_title('GMM Model Selection — BIC & AIC', fontsize=13)
ax.set_xlabel('Number of Components'); ax.set_ylabel('Score (lower = better)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out('GMM_bic_aic.png'), dpi=150)
plt.show()

---
## 🎯 Step 4 — Fit Final GMM

We fit GMM with the BIC-optimal number of components (or override to 2 for clean declustering).

| `N_COMPONENTS` | Interpretation |
|---|---|
| 2 | Strict crisis / non-crisis (cleanest comparison with KDM) |
| BIC optimal | Sub-types: e.g. aftershocks, swarms, background, deep events |

Set `FORCE_2_COMPONENTS = True` to force 2 components for direct KDM comparison.

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────
FORCE_2_COMPONENTS = False   # True = force 2 for KDM comparison
N_COMPONENTS = 2 if FORCE_2_COMPONENTS else best_bic_n
# ─────────────────────────────────────────────────────────

print(f'Fitting GMM with {N_COMPONENTS} components on {len(X_pca):,} events...')
gmm_final = GaussianMixture(
    n_components    = N_COMPONENTS,
    covariance_type = COVARIANCE_TYPE,
    random_state    = 42,
    max_iter        = 300,
    n_init          = 5,
    verbose         = 1
)
gmm_final.fit(X_pca)
print(f'\n✅ GMM converged: {gmm_final.converged_}')
print(f'   Log-likelihood : {gmm_final.lower_bound_:.4f}')

---
## 🏷️ Step 5 — Assign Crisis / Non-Crisis Labels

GMM gives us **posterior probabilities** for each event in each component.
We need to map GMM components → crisis / non-crisis using physical reasoning:

```
Component with smaller mean T + smaller mean R → CRISIS (events clustered in space-time)
Component with larger  mean T + larger  mean R → NON-CRISIS (isolated background)
```

In [ ]:
# Get probabilities and hard labels
proba       = gmm_final.predict_proba(X_pca)   # shape: (n_events, n_components)
hard_labels = gmm_final.predict(X_pca)

# Map components to crisis/non-crisis by comparing mean T distances
t_indices = [feature_cols.index(c) for c in t_cols]
r_indices = [feature_cols.index(c) for c in r_cols]

# For each GMM component, compute mean T and R in original (unscaled) space
# Use component means projected back (approximate via event averages)
comp_T_means = []
for comp in range(N_COMPONENTS):
    mask = hard_labels == comp
    if mask.sum() > 0:
        comp_T_means.append(X[mask][:, t_indices].mean())
    else:
        comp_T_means.append(np.inf)

# Smallest mean T = crisis component (close in time to neighbours)
crisis_comp     = np.argmin(comp_T_means)
non_crisis_comp = np.argmax(comp_T_means)
print(f'Component {crisis_comp} → CRISIS     (mean T = {comp_T_means[crisis_comp]:.4f})')
print(f'Component {non_crisis_comp} → NON-CRISIS (mean T = {comp_T_means[non_crisis_comp]:.4f})')

# If more than 2 components, assign remaining by T-distance ranking
comp_ranking = np.argsort(comp_T_means)   # sorted small → large T
crisis_comps     = set(comp_ranking[:N_COMPONENTS//2])
non_crisis_comps = set(comp_ranking[N_COMPONENTS//2:])

# Build P_crisis: sum of probabilities from crisis components
P_crisis = proba[:, list(crisis_comps)].sum(axis=1)
P_non_crisis = 1.0 - P_crisis

df_labelled = df_clean.copy()
df_labelled['gmm_component'] = hard_labels
df_labelled['P_crisis']      = P_crisis
df_labelled['P_non_crisis']  = P_non_crisis
df_labelled['label']         = np.where(P_crisis >= 0.5, 'crisis', 'non_crisis')
df_labelled['confidence']    = np.abs(0.5 - df_labelled['P_crisis'].clip(0,1)) / 0.5

# Add individual component probs
for c in range(N_COMPONENTS):
    df_labelled[f'P_comp_{c}'] = proba[:, c]

n_c  = (df_labelled['label']=='crisis').sum()
n_nc = (df_labelled['label']=='non_crisis').sum()
print(f'\n── GMM Classification ──────────────────────────')
print(f'  Total      : {len(df_labelled):,}')
print(f'  Crisis     : {n_c:,}  ({100*n_c/len(df_labelled):.1f}%)')
print(f'  Non-crisis : {n_nc:,}  ({100*n_nc/len(df_labelled):.1f}%)')
print(f'────────────────────────────────────────────────')

---
## 📈 Step 6 — Visualisations
### Plot A — GMM Component Ellipses in PCA Space

In [ ]:
def draw_ellipse(position, covariance, ax, color, alpha=0.3, n_std=2.0):
    """Draw a 2D covariance ellipse for a GMM component."""
    vals, vecs = np.linalg.eigh(covariance[:2,:2])
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:,order]
    theta = np.degrees(np.arctan2(*vecs[:,0][::-1]))
    w, h  = 2 * n_std * np.sqrt(vals)
    ell   = Ellipse(xy=position[:2], width=w, height=h,
                    angle=theta, color=color, alpha=alpha)
    ax.add_patch(ell)

fig, axes = plt.subplots(1,2, figsize=(16,7))

# Use PCA 1 & 2 for plotting
colors = plt.cm.tab10(np.linspace(0,1,N_COMPONENTS))

for ax_idx, ax in enumerate(axes):
    # Subsample for speed
    idx_plot = np.random.choice(len(X_pca), min(30000,len(X_pca)), replace=False)
    for comp in range(N_COMPONENTS):
        mask  = hard_labels[idx_plot] == comp
        ctype = 'crisis' if comp in crisis_comps else 'non-crisis'
        if ax_idx == 0:
            ax.scatter(X_pca[idx_plot][mask,0], X_pca[idx_plot][mask,1],
                       s=0.5, alpha=0.4, color=colors[comp],
                       label=f'Comp {comp} ({ctype})')
            if gmm_final.covariance_type == 'full':
                draw_ellipse(gmm_final.means_[comp], gmm_final.covariances_[comp],
                             ax, color=colors[comp])
    if ax_idx == 0:
        ax.set_title(f'GMM Components in PCA Space ({N_COMPONENTS} components)', fontsize=12)
        ax.set_xlabel('PC-1'); ax.set_ylabel('PC-2')
        ax.legend(markerscale=8, fontsize=8)
    else:
        sc = ax.scatter(X_pca[idx_plot,0], X_pca[idx_plot,1],
                        c=P_crisis[idx_plot], cmap='RdYlGn_r',
                        s=0.5, alpha=0.5, vmin=0, vmax=1)
        plt.colorbar(sc, ax=ax, label='P(crisis)')
        ax.set_title('P(crisis) in PCA Space', fontsize=12)
        ax.set_xlabel('PC-1'); ax.set_ylabel('PC-2')

plt.suptitle('GMM Declustering — NZ Earthquake Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(out('GMM_pca_components.png'), dpi=150)
plt.show()

### Plot B — Spatial Map + Cumulative Curves

In [ ]:
crisis     = df_labelled[df_labelled['label']=='crisis']
non_crisis = df_labelled[df_labelled['label']=='non_crisis']
time_col   = 'time' if 'time' in df_labelled.columns else 'Year'

fig, axes = plt.subplots(1,3, figsize=(20,6))

# Spatial
axes[0].scatter(non_crisis['longitude'], non_crisis['latitude'],
                c='steelblue', s=0.5, alpha=0.3, label=f'Non-crisis ({len(non_crisis):,})')
axes[0].scatter(crisis['longitude'],     crisis['latitude'],
                c='crimson',   s=0.5, alpha=0.3, label=f'Crisis ({len(crisis):,})')
axes[0].set_title('GMM — Spatial Classification', fontsize=11)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(markerscale=8)

# P(crisis) map
sc = axes[1].scatter(df_labelled['longitude'], df_labelled['latitude'],
                     c=df_labelled['P_crisis'], cmap='RdYlGn_r',
                     s=0.5, alpha=0.4, vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label='P(crisis)')
axes[1].set_title('GMM — P(crisis) Map', fontsize=11)
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

# Cumulative
df_s = df_labelled.sort_values(time_col)
N    = len(df_s)
c_s  = df_s[df_s['label']=='crisis']
nc_s = df_s[df_s['label']=='non_crisis']
axes[2].plot(df_s[time_col],   np.arange(N)/N, 'k--', lw=1, alpha=0.5, label='Full catalogue')
axes[2].plot(c_s[time_col],   np.arange(len(c_s))/N, 'r:', lw=1.5, label=f'Crisis ({len(c_s):,})')
axes[2].plot(nc_s[time_col],  np.arange(len(nc_s))/N, 'b-', lw=1.5, label=f'Non-crisis ({len(nc_s):,})')
axes[2].set_title('Cumulative Curves — GMM', fontsize=11)
axes[2].set_xlabel('Time (decimal years)'); axes[2].set_ylabel('Proportion')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('GMM Declustering Results — NZ Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(out('GMM_results_overview.png'), dpi=150)
plt.show()

### Plot C — Probability Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(18,5))

# P(crisis) histogram
axes[0].hist(df_labelled['P_crisis'], bins=50, color='crimson', alpha=0.7, edgecolor='white')
axes[0].axvline(0.5, color='black', linestyle='--', label='Threshold=0.5')
axes[0].set_title('P(crisis) Distribution — GMM', fontsize=11)
axes[0].set_xlabel('P(crisis)'); axes[0].set_ylabel('Count')
axes[0].legend()

# Confidence distribution
axes[1].hist(df_labelled['confidence'], bins=50, color='purple', alpha=0.7, edgecolor='white')
axes[1].set_title('Classification Confidence', fontsize=11)
axes[1].set_xlabel('Confidence (0=uncertain, 1=certain)'); axes[1].set_ylabel('Count')

# Magnitude
bins = np.arange(0, df_labelled['magnitude'].max()+0.5, 0.5)
axes[2].hist(crisis['magnitude'],     bins=bins, density=True, alpha=0.6, color='orange',    label='Crisis')
axes[2].hist(non_crisis['magnitude'], bins=bins, density=True, alpha=0.6, color='steelblue', label='Non-crisis')
axes[2].set_title('Magnitude Distribution — GMM', fontsize=11)
axes[2].set_xlabel('Magnitude'); axes[2].set_ylabel('Density')
axes[2].legend()

plt.tight_layout()
plt.savefig(out('GMM_probability_analysis.png'), dpi=150)
plt.show()

---
## 💾 Step 7 — Save Results

In [ ]:
save_cols = [c for c in ['event','DateTime','latitude','longitude','depth',
             'magnitude','time','gmm_component','P_crisis','P_non_crisis',
             'label','confidence'] if c in df_labelled.columns]
# Also add individual component columns
comp_cols = [f'P_comp_{c}' for c in range(N_COMPONENTS)]
save_cols += comp_cols
df_labelled[save_cols].to_csv(out('NZ_GMM_declustered.csv'), index=False)

n_c  = (df_labelled['label']=='crisis').sum()
n_nc = (df_labelled['label']=='non_crisis').sum()
print('✅ Saved → NZ_GMM_declustered.csv')
print(f'\n══ GMM Final Summary ══════════════════════════')
print(f'  Total       : {len(df_labelled):,}')
print(f'  Crisis      : {n_c:,}  ({100*n_c/len(df_labelled):.1f}%)')
print(f'  Non-crisis  : {n_nc:,}  ({100*n_nc/len(df_labelled):.1f}%)')
print(f'  Components  : {N_COMPONENTS}  (BIC optimal: {best_bic_n})')
print(f'  Converged   : {gmm_final.converged_}')
print(f'══════════════════════════════════════════════')